# Stage 4 — Fetch all 3 datasets to personal Drive (`My Drive/pnid_extraction`)

**Purpose:** run the Colab *compute* (GPU) under whichever account you like (e.g. a shared
account for quota reasons), while mounting **your own personal Google Drive** for storage —
these are independent auth steps in Colab, so the account running the notebook does not have
to be the account you mount.

When the `drive.mount()` cell below prompts for auth, sign in with your **personal** Google
account, not whichever account is running this Colab session.

Downloads directly from source — no manual upload, no `files.upload()` widget:
1. **Gupta PID_Dataset** (real, CC-BY-4.0) — `zenodo.org/records/8028570`
2. **Kaggle P&ID Symbols** (synthetic, fine-tune volume) — `kaggle.com/datasets/hristohristov21/pid-symbols`
3. **PID2Graph OPEN100** (real, connectivity graph ground truth) — `zenodo.org/records/14803338`

All three land under `/content/drive/MyDrive/pnid_extraction/` in your personal Drive.

## 1. Mount Drive (personal account)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ROOT = Path("/content/drive/MyDrive/pnid_extraction")
ROOT.mkdir(parents=True, exist_ok=True)
print(f"Target folder: {ROOT}")

## 2. Gupta PID_Dataset (Zenodo, real ground truth)

6.7GB zip. Streams straight to Drive — Zenodo supports range requests so this resumes if
interrupted (rerun the cell; `wget -c` continues a partial download rather than restarting).

In [ ]:
GUPTA_DIR = ROOT / "gupta_pid"
GUPTA_DIR.mkdir(parents=True, exist_ok=True)
GUPTA_ZIP = GUPTA_DIR / "PID_Dataset.zip"

!wget -c -O "{GUPTA_ZIP}" "https://zenodo.org/records/8028570/files/PID_Dataset.zip?download=1"

import hashlib
def md5sum(path, chunk=8*1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while (b := f.read(chunk)):
            h.update(b)
    return h.hexdigest()

print("MD5:", md5sum(GUPTA_ZIP))
print("Compare against the checksum on the Zenodo record page before trusting this download.")

In [ ]:
import zipfile
with zipfile.ZipFile(GUPTA_ZIP) as zf:
    zf.extractall(GUPTA_DIR)
print("Extracted to", GUPTA_DIR)
print(f"Top-level entries: {list(GUPTA_DIR.iterdir())[:10]}")

## 3. Kaggle P&ID Symbols (synthetic, fine-tune volume)

Needs a Kaggle API token (`kaggle.json`). Upload yours via the file picker when prompted, or
if you already have `KAGGLE_USERNAME`/`KAGGLE_KEY` as Colab secrets, skip the upload cell and
set the env vars directly instead.

In [ ]:
import os
from pathlib import Path

KAGGLE_DIR_CFG = Path("/root/.kaggle")
KAGGLE_DIR_CFG.mkdir(parents=True, exist_ok=True)

if not (KAGGLE_DIR_CFG / "kaggle.json").exists():
    from google.colab import files
    print("Upload your kaggle.json (Kaggle account -> Settings -> Create New Token):")
    uploaded = files.upload()
    for fname in uploaded:
        Path(fname).rename(KAGGLE_DIR_CFG / "kaggle.json")

os.chmod(KAGGLE_DIR_CFG / "kaggle.json", 0o600)
!pip install -q kaggle

In [ ]:
KAGGLE_DIR = ROOT / "kaggle_pid_symbols"
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d hristohristov21/pid-symbols -p "{KAGGLE_DIR}" --unzip

n_files = sum(1 for _ in KAGGLE_DIR.rglob("*") if _.is_file())
print(f"Kaggle dataset: {n_files} files under {KAGGLE_DIR}")
print("Expected per dataset card: 30,000 labeled images, 195,759 instances, 32 classes.")

## 4. PID2Graph OPEN100 (Zenodo, real connectivity ground truth)

9.3GB zip — same resumable-download + checksum pattern as Gupta.

In [ ]:
PID2GRAPH_DIR = ROOT / "pid2graph"
PID2GRAPH_DIR.mkdir(parents=True, exist_ok=True)
PID2GRAPH_ZIP = PID2GRAPH_DIR / "OPEN100.zip"

!wget -c -O "{PID2GRAPH_ZIP}" "https://zenodo.org/records/14803338/files/OPEN100.zip?download=1"

print("MD5:", md5sum(PID2GRAPH_ZIP))
print("Compare against the checksum on the Zenodo record page before trusting this download.")

In [ ]:
with zipfile.ZipFile(PID2GRAPH_ZIP) as zf:
    zf.extractall(PID2GRAPH_DIR)
print("Extracted to", PID2GRAPH_DIR)
print(f"Top-level entries: {list(PID2GRAPH_DIR.iterdir())[:10]}")

## 5. Final check — all three present under `pnid_extraction`

In [ ]:
for name, d in [("Gupta PID_Dataset", GUPTA_DIR), ("Kaggle P&ID Symbols", KAGGLE_DIR), ("PID2Graph OPEN100", PID2GRAPH_DIR)]:
    n = sum(1 for _ in d.rglob("*") if _.is_file())
    size_gb = sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / 1e9
    print(f"{name:24s} {n:6d} files   {size_gb:6.2f} GB   {d}")